In [12]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [13]:
import numpy as np
import pandas as pd
from collections import defaultdict
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
import tensorflow.keras.backend as K
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import RandomOverSampler

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import tensorflow as tf
import tensorflow_hub as hub
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

data_path = '/kaggle/input/datasets/tushaamagrawal/icbhi-2017-challenge/ICBHI_final_database'
total_files = os.listdir(data_path)

print("Total files are:", len(total_files))
wav_files = [f for f in total_files if f.endswith(".wav")]
txt_files = [f for f in total_files if f.endswith(".txt")]

wav_set = set([f.replace(".wav", "") for f in wav_files])
txt_set = set([f.replace(".txt", "") for f in txt_files])
common_files = list(wav_set.intersection(txt_set))

print("Usable common files:", len(common_files))

train_files, test_files = train_test_split(common_files, test_size=0.2, random_state=42)
train_files, val_files  = train_test_split(train_files, test_size=0.2, random_state=42)

Total files are: 1843
Usable common files: 920


In [15]:
def add_noise(signal, noise_factor=0.005):
    noise = np.random.randn(len(signal))
    return signal + noise_factor * noise

def time_shift(signal, shift_max=0.2):
    shift = int(np.random.uniform(-shift_max, shift_max) * len(signal))
    return np.roll(signal, shift)

def time_stretch(signal, rate=0.9):
    return librosa.effects.time_stretch(signal, rate=rate)

def pitch_shift(signal, sr=16000, n_steps=1):
    return librosa.effects.pitch_shift(signal, sr=sr, n_steps=n_steps)

def augment_signal(signal, sr=16000):
    augmented = [signal] # Original basic pattern
    augmented.append(add_noise(signal))
    augmented.append(time_shift(signal))
    try:
        augmented.append(time_stretch(signal, rate=np.random.uniform(0.9, 1.1)))
    except:
        pass
    try:
        augmented.append(pitch_shift(signal, sr, n_steps=np.random.uniform(-1, 1)))
    except:
        pass
    return augmented

In [16]:
def build_segments(file_list, augment=False, return_ids=False):
    segments = []
    labels = []
    file_ids = []

    for file in file_list:
        audio_path = f"{data_path}/{file}.wav"
        text_path  = f"{data_path}/{file}.txt"

        y, sr = librosa.load(audio_path, sr=16000)

        with open(text_path, "r") as f:
            for line in f:
                start, end, crackle, wheeze = line.strip().split()
                start, end = float(start), float(end)
                crackle, wheeze = int(crackle), int(wheeze)

                segment = y[int(start*sr):int(end*sr)]
                if len(segment) < 1000:
                    continue

                # 4-class labeling mechanism
                if crackle == 0 and wheeze == 0:
                    label = 0  # Normal
                elif crackle == 1 and wheeze == 0:
                    label = 1  # Crackle
                elif crackle == 0 and wheeze == 1:
                    label = 2  # Wheeze
                else:
                    label = 3  # Both Crackle and Wheeze

                if augment:
                    aug_list = augment_signal(segment, sr)
                    for aug in aug_list:
                        segments.append(aug)
                        labels.append(label)
                        file_ids.append(file)
                else:
                    segments.append(segment)
                    labels.append(label)
                    file_ids.append(file)

    if return_ids:
        return segments, labels, file_ids
    return segments, labels

In [17]:

import zipfile
from collections import defaultdict
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

print("Segmenting and processing datasets...")
X_train, y_train = build_segments(train_files, augment=False) 
X_val, y_val     = build_segments(val_files, augment=False)
X_test, y_test, test_ids = build_segments(test_files, augment=False, return_ids=True)

file_true = {}
for label, fid in zip(y_test, test_ids):
    file_true[fid] = label

Segmenting and processing datasets...


In [18]:
yamnet = hub.load("https://tfhub.dev/google/yamnet/1")

def extract_statistical_pooling(signal):
    signal = signal.astype(np.float32)
    _, embeddings, _ = yamnet(signal)
    emb = embeddings.numpy()

    mean   = np.mean(emb, axis=0)
    std    = np.std(emb, axis=0)
    max_   = np.max(emb, axis=0)
    min_   = np.min(emb, axis=0)
    median = np.median(emb, axis=0)
    q25    = np.percentile(emb, 25, axis=0)
    q75    = np.percentile(emb, 75, axis=0)

    return np.concatenate([mean, std, max_, min_, median, q25, q75])    
    
def create_embedding_dataset(X, y):
    X_emb = []
    y_clean = []
    for seg, label in zip(X, y):
        try:
            emb = extract_statistical_pooling(seg)
            X_emb.append(emb)
            y_clean.append(label)
        except:
            continue
    return np.array(X_emb), np.array(y_clean)

In [19]:
X_train_emb, y_train = create_embedding_dataset(X_train, y_train)
X_val_emb, y_val     = create_embedding_dataset(X_val, y_val)
X_test_emb, y_test   = create_embedding_dataset(X_test, y_test)

In [20]:
def reset_and_create_mlp(input_dim=7168):
    """ Clears global backend graph and returns a fresh weight-initialized instance """
    tf.keras.backend.clear_session()
    tf.random.set_seed(42) # Strict reproducibility anchor
    np.random.seed(42)
    
    model = Sequential([
        Dense(2048, activation='relu', input_shape=(input_dim,)),
        Dropout(0.3),
        Dense(512, activation='relu'),
        Dropout(0.3),
        Dense(256, activation='relu'),
        Dropout(0.2),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dropout(0.3),
        Dense(16, activation='relu'),
        Dropout(0.3),
        Dense(8, activation='relu'),
        Dropout(0.3),
        Dense(4, activation='softmax')
    ])
    return model

In [21]:
def run_file_level_evaluation(trained_model, X_test_features, y_test_labels, test_file_ids, shared_file_truth):
    """ Generates predictions using the Top-K Weighted File-Level Aggregation algorithm """
    y_pred_prob = trained_model.predict(X_test_features, verbose=0)
    
    file_preds = defaultdict(list)
    for prob, label, fid in zip(y_pred_prob, y_test_labels, test_file_ids):
        file_preds[fid].append(prob)
        
    y_true_file = []
    y_pred_file = []
    
    for fid in file_preds:
        probs = np.array(file_preds[fid])
        conf = np.max(probs, axis=1)
        k = max(1, len(probs)//3)
        top_idx = np.argsort(conf)[-k:]
        top_probs = probs[top_idx]
        top_conf = conf[top_idx]
        weights = top_conf / np.sum(top_conf)
        final_prob = np.sum(top_probs * weights[:, None], axis=0)
        
        y_pred_file.append(np.argmax(final_prob))
        y_true_file.append(shared_file_truth[fid])
        
    acc = accuracy_score(y_true_file, y_pred_file)
    macro_f1 = f1_score(y_true_file, y_pred_file, average='macro')
    weighted_f1 = f1_score(y_true_file, y_pred_file, average='weighted')
    
    return acc, macro_f1, weighted_f1

FIXED_LR = 0.0001
FIXED_EPOCHS = 50
FIXED_BATCH_SIZE = 32
final_paper_records = []

In [22]:
class_counts = np.bincount(y_train)
total_samples = len(y_train)
num_classes = len(class_counts)

# SQUARE ROOT SMOOTHING: prevents minority penalties from over-biasing internal nodes
smoothed_weights = total_samples / (num_classes * np.power(class_counts, 0.5))
smoothed_weights = smoothed_weights / np.mean(smoothed_weights) # Scale normalizing bounds
smoothed_class_weights_dict = {i: float(smoothed_weights[i]) for i in range(num_classes)}

cw_model = reset_and_create_mlp(input_dim=X_train_emb.shape[1])
cw_model.compile(optimizer=Adam(learning_rate=FIXED_LR), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

cw_model.fit(
    X_train_emb, y_train, 
    validation_data=(X_val_emb, y_val), 
    epochs=FIXED_EPOCHS, 
    batch_size=FIXED_BATCH_SIZE, 
    class_weight=smoothed_class_weights_dict, 
    verbose=0
)
acc_cw, mf1_cw, wf1_cw = run_file_level_evaluation(cw_model, X_test_emb, y_test, test_ids, file_true)

final_paper_records.append({
    "Balancing Technique": "Smoothed Class Weights",
    "Accuracy": f"{acc_cw*100:.2f}%",
    "Macro F1": f"{mf1_cw:.4f}",
    "Weighted F1": f"{wf1_cw:.4f}"
})

# ==========================================
# APPROACH 3: MATHEMATICAL FOCAL LOSS OBJECTIVE
# ==========================================
print("\n▶️ [3/4] Initializing Multi-Class Focal Loss optimization loops...")
def multi_class_focal_loss(gamma=2.0, alpha=0.25):
    """ Direct tensor wrapper to dynamically suppress gradients from well-classified tokens """
    def loss_calculation(y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        y_true_one_hot = tf.one_hot(y_true, depth=4)
        y_pred = K.clip(y_pred, K.epsilon(), 1.0 - K.epsilon())
        
        cross_entropy = -y_true_one_hot * K.log(y_pred)
        modulating_weight = y_true_one_hot * K.pow(1.0 - y_pred, gamma)
        
        return K.sum(alpha * modulating_weight * cross_entropy, axis=-1)
    return loss_calculation

fl_model = reset_and_create_mlp(input_dim=X_train_emb.shape[1])
fl_model.compile(optimizer=Adam(learning_rate=FIXED_LR), loss=multi_class_focal_loss(gamma=2.0, alpha=0.25), metrics=['accuracy'])

fl_model.fit(X_train_emb, y_train, validation_data=(X_val_emb, y_val), epochs=FIXED_EPOCHS, batch_size=FIXED_BATCH_SIZE, verbose=0)
acc_fl, mf1_fl, wf1_fl = run_file_level_evaluation(fl_model, X_test_emb, y_test, test_ids, file_true)

final_paper_records.append({
    "Balancing Technique": "Focal Loss Regularization",
    "Accuracy": f"{acc_fl*100:.2f}%",
    "Macro F1": f"{mf1_fl:.4f}",
    "Weighted F1": f"{wf1_fl:.4f}"
})

# ==========================================
# APPROACH 4: VECTOR EMBEDDING OVERSAMPLING
# ==========================================
print("\n▶️ [4/4] Executing Synthetic Feature Space Oversampling...")
ros = RandomOverSampler(random_state=42)
X_train_resampled, y_train_resampled = ros.fit_resample(X_train_emb, y_train)

oversampled_model = reset_and_create_mlp(input_dim=X_train_emb.shape[1])
oversampled_model.compile(optimizer=Adam(learning_rate=FIXED_LR), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

oversampled_model.fit(X_train_resampled, y_train_resampled, validation_data=(X_val_emb, y_val), epochs=FIXED_EPOCHS, batch_size=FIXED_BATCH_SIZE, verbose=0)
acc_os, mf1_os, wf1_os = run_file_level_evaluation(oversampled_model, X_test_emb, y_test, test_ids, file_true)

final_paper_records.append({
    "Balancing Technique": "Random Oversampling",
    "Accuracy": f"{acc_os*100:.2f}%",
    "Macro F1": f"{mf1_os:.4f}",
    "Weighted F1": f"{wf1_os:.4f}"
})

# ==========================================
# 3. COMPILING REPORT DATAFRAME
# ==========================================


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1780254124.300743     148 service.cc:152] XLA service 0x7e1094034980 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1780254124.300786     148 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1780254124.300791     148 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1780254128.888152     148 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.



▶️ [3/4] Initializing Multi-Class Focal Loss optimization loops...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



▶️ [4/4] Executing Synthetic Feature Space Oversampling...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



📊 Compiled Imbalance Analysis Grid Matrix for Conference Submission:
| Balancing Technique       | Accuracy   |   Macro F1 |   Weighted F1 |
|:--------------------------|:-----------|-----------:|--------------:|
| Smoothed Class Weights    | 63.04%     |     0.4041 |        0.6184 |
| Focal Loss Regularization | 62.50%     |     0.2166 |        0.4931 |
| Random Oversampling       | 52.17%     |     0.4168 |        0.5544 |


In [23]:
df_paper_table = pd.DataFrame(final_paper_records)
print("\n📊 Analysis grid:")
print(df_paper_table.to_markdown(index=False))


📊 Analysis grid:
| Balancing Technique       | Accuracy   |   Macro F1 |   Weighted F1 |
|:--------------------------|:-----------|-----------:|--------------:|
| Smoothed Class Weights    | 63.04%     |     0.4041 |        0.6184 |
| Focal Loss Regularization | 62.50%     |     0.2166 |        0.4931 |
| Random Oversampling       | 52.17%     |     0.4168 |        0.5544 |
